# PAI最終課題：日本語指示で目印を巡回する移動ロボット（MuJoCo × ローカルLLM）

このノート1冊で、環境構築 → ロボットの動作確認 → LLM接続 → 動画の仕上げ、まで全部やります。**上から順に実行**してください。

| 章 | やること | できあがるもの | 目安 |
|---|---|---|---|
| 0 | MuJoCoを入れて、ロボットが目印まで走る動画を1本 | 動く環境 ＝ **合格要件** | 10分 |
| 1 | Qwen3.5 で日本語指示 → 行動計画 → 走行 | 指示どおりに動く動画 ＝ 加点 | 40分 |
| 2 | タイトル・字幕を付けて1本にまとめる | 提出用 mp4 | 15分 |

**構成**：日本語の指示 → LLM が `[{"action":"goto","target":"red_pillar"}, {"action":"turn","deg":90}, ...]` という記号的な計画を出す → 検証層（存在しない目印・範囲外の値をはじく）→ 古典制御（目標への向きを合わせて進む）が差動二輪ロボットを動かす → 追従カメラで録画。
LLMは計画だけを担当し、移動は決定的な制御が担うので、LLMの誤りがロボットの暴走に直結しません。

**ランタイム**：GPU（T4）があればLLMに Qwen3.5-4B を使います。GPUが取れなくても、CPUで小型モデル（Qwen3.5-0.8B）に自動で切り替わるので最後まで通せます。
MuJoCo自体はCPUで動き、描画もGPU不要です。

**切断されたとき・ランタイムを再起動したとき**：動画と計画は Google Drive の `PAI_final/` に保存されます。再接続後は 0-1 → 0-2 → 0-3 を実行し直してから続きへ。
**描画の設定（`MUJOCO_GL`）は mujoco を最初に import する前に決める必要がある**ので、0-1 より前に mujoco を import するセルを足さないでください。


## 0. 環境構築と動作確認

### 0-1. インストール（毎回必要・約1分）
- `mujoco`：物理シミュレータ（DeepMind製、pipで入る）
- `imageio` `imageio-ffmpeg`：mp4の書き出し
- `libosmesa6`：画面のない環境でCPU描画するためのライブラリ（GPUがある場合はEGLで描画し、なければこちらに自動で切り替え）


In [ ]:
%pip install -q mujoco imageio imageio-ffmpeg
!apt-get -qq install -y libosmesa6 libosmesa6-dev fonts-noto-cjk ffmpeg > /dev/null 2>&1

# 描画バックエンドの指定は「mujoco を最初に import する前」に必要（import 後に変えても効かない）
import os, torch
os.environ["MUJOCO_GL"] = "egl" if torch.cuda.is_available() else "osmesa"
import mujoco
print("mujoco", mujoco.__version__, "/ GPU:", torch.cuda.is_available(), "/ 描画:", os.environ["MUJOCO_GL"])


### 0-2. Google Driveをつなぐ（毎回必要）
動画と計画を `MyDrive/PAI_final/` に保存します。既存のファイルには触れません。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE = "/content/drive/MyDrive/PAI_final"
for d in ["videos", "plans"]:
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)
print("保存先:", DRIVE)


### 0-3. ロボットとシミュレーションの定義（`rover_sim.py` として書き出す）
このセルは、ロボットの定義（MJCFというMuJoCoのXML）と制御のコードを `rover_sim.py` に書き出します。そのままGitHubに置けるファイルです。中身は次のとおりです。

- **場面**：床と4つの目印（赤い柱・青い柱・緑の箱・黄色い箱）。目印の名前と位置は `LANDMARKS` にまとめてあり、LLMにもこの名前を渡します
- **ロボット**：箱型の車体、左右の車輪（速度制御のモーター）、後ろのキャスター。車輪半径0.08 m、車輪間隔0.30 m
- **低レベル**：`drive(v, w)` 前進速度と旋回速度を、左右の車輪の回転速度に変換（差動二輪の運動学）
- **中レベル**：`goto(目印)` 目印の方向と自分の向きの差に比例して旋回し、向きが合っていれば前進。目印の手前で停止。`turn(角度)`、`wait(秒)`
- **高レベル**：`execute(計画)` 計画を順に実行し、各行動の開始・終了時刻と到達の成否を記録（レポート用）
- **描画**：`chase` という追従カメラの映像を毎秒25コマ記録し、`save_video()` でmp4に保存

描画のバックエンド（EGL か OSMesa）は環境に合わせて自動で選びます。


In [ ]:
import os
assert os.environ.get('MUJOCO_GL') in ('egl', 'osmesa'), '0-1 を先に実行してください（描画設定が未設定）'

rover_src = r'''"""
rover_sim.py — MuJoCo 上の差動二輪ロボットと、目印へ向かう簡単な制御・録画

構成
  - 床、色つきの目印（赤い柱・青い柱・緑の箱・黄色い箱）
  - 差動二輪ロボット：左右の車輪を速度制御、後ろにキャスター（球）
  - 行動計画 = [{"action":"goto","target":"red_pillar"}, {"action":"turn","deg":90}, {"action":"wait","sec":1}, ...]
    を順に実行し、追従カメラの映像を mp4 に保存する
"""
import math
import numpy as np
import mujoco
import imageio

# 目印の名前（LLMにもこの名前を渡す）と、位置・色
LANDMARKS = {
    "red_pillar":   {"pos": (3.0, 0.0),   "rgba": "0.9 0.2 0.2 1", "shape": "cylinder", "radius": 0.12, "ja": "赤い柱"},
    "blue_pillar":  {"pos": (0.0, 3.0),   "rgba": "0.2 0.3 0.9 1", "shape": "cylinder", "radius": 0.12, "ja": "青い柱"},
    "green_box":    {"pos": (-3.0, 0.0),  "rgba": "0.2 0.7 0.3 1", "shape": "box",      "radius": 0.36, "ja": "緑の箱"},
    "yellow_box":   {"pos": (0.0, -3.0),  "rgba": "0.9 0.8 0.2 1", "shape": "box",      "radius": 0.36, "ja": "黄色い箱"},
}

WHEEL_R = 0.08     # 車輪半径 [m]
WHEEL_L = 0.30     # 左右車輪の間隔 [m]
V_MAX = 0.6        # 最大前進速度 [m/s]
W_MAX = 1.5        # 最大旋回速度 [rad/s]


def build_xml():
    marks = []
    for name, m in LANDMARKS.items():
        x, y = m["pos"]
        if m["shape"] == "cylinder":
            geom = f'<geom name="{name}" type="cylinder" size="0.12 0.5" pos="{x} {y} 0.5" rgba="{m["rgba"]}"/>'
        else:
            geom = f'<geom name="{name}" type="box" size="0.25 0.25 0.25" pos="{x} {y} 0.25" rgba="{m["rgba"]}"/>'
        marks.append(geom)
    marks = "\n      ".join(marks)
    return f"""
<mujoco model="rover">
  <option timestep="0.002" gravity="0 0 -9.81" integrator="implicitfast"/>
  <visual><global offwidth="640" offheight="480"/><headlight diffuse="0.3 0.3 0.3" ambient="0.3 0.3 0.3" specular="0 0 0"/></visual>
  <asset>
    <texture name="grid" type="2d" builtin="checker" rgb1="0.55 0.58 0.62" rgb2="0.42 0.45 0.5" width="512" height="512"/>
    <material name="grid" texture="grid" texrepeat="12 12" reflectance="0.05"/>
  </asset>
  <default>
    <geom condim="3" friction="1.0 0.005 0.0001"/>
  </default>
  <worldbody>
    <light pos="0 0 6" dir="0.2 0.2 -1" directional="true" diffuse="0.5 0.5 0.5"/>
    <geom name="floor" type="plane" size="8 8 0.1" material="grid"/>
    {marks}
    <body name="chassis" pos="0 0 0.12">
      <freejoint name="root"/>
      <geom name="body" type="box" size="0.18 0.12 0.04" rgba="0.25 0.3 0.4 1" mass="3.0"/>
      <geom name="nose" type="box" size="0.03 0.06 0.02" pos="0.19 0 0.03" rgba="1 0.5 0 1" mass="0.05"/>
      <site name="center" pos="0 0 0" size="0.01"/>
      <camera name="chase" mode="trackcom" pos="-2.2 0 1.4" xyaxes="0 -1 0 0.5 0 1"/>
      <body name="wheel_l" pos="0 {WHEEL_L/2} -0.02">
        <joint name="jl" type="hinge" axis="0 1 0" armature="0.01" damping="0.05"/>
        <geom type="cylinder" size="{WHEEL_R} 0.02" euler="90 0 0" rgba="0.1 0.1 0.1 1" mass="0.3"/>
      </body>
      <body name="wheel_r" pos="0 {-WHEEL_L/2} -0.02">
        <joint name="jr" type="hinge" axis="0 1 0" armature="0.01" damping="0.05"/>
        <geom type="cylinder" size="{WHEEL_R} 0.02" euler="90 0 0" rgba="0.1 0.1 0.1 1" mass="0.3"/>
      </body>
      <geom name="caster" type="sphere" size="0.04" pos="-0.15 0 -0.06" rgba="0.5 0.5 0.5 1" mass="0.1" friction="0.01 0.001 0.0001"/>
    </body>
  </worldbody>
  <actuator>
    <velocity name="ml" joint="jl" kv="1.0" ctrlrange="-20 20"/>
    <velocity name="mr" joint="jr" kv="1.0" ctrlrange="-20 20"/>
  </actuator>
</mujoco>
"""


_MODEL, _RENDERER = None, None   # 描画器は1プロセスに1つだけ作る（複数作ると画面が乱れる環境があるため）


class Rover:
    def __init__(self, width=640, height=480, fps=25):
        global _MODEL, _RENDERER
        if _MODEL is None:
            _MODEL = mujoco.MjModel.from_xml_string(build_xml())
            _RENDERER = mujoco.Renderer(_MODEL, height, width)
        self.model = _MODEL
        self.renderer = _RENDERER
        self.data = mujoco.MjData(self.model)
        self.cam = mujoco.MjvCamera()
        self.cam.type = mujoco.mjtCamera.mjCAMERA_FIXED
        self.cam.fixedcamid = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_CAMERA, "chase")
        self.fps = fps
        self.steps_per_frame = int(round(1.0 / (fps * self.model.opt.timestep)))
        self.frames = []
        self.trace = []           # (t, x, y, yaw) の記録。レポート用
        self.t = 0.0
        mujoco.mj_forward(self.model, self.data)

    # --- 状態 ---
    def pose(self):
        x, y = self.data.qpos[0], self.data.qpos[1]
        w, qx, qy, qz = self.data.qpos[3:7]
        yaw = math.atan2(2 * (w * qz + qx * qy), 1 - 2 * (qy * qy + qz * qz))
        return x, y, yaw

    # --- 低レベル：速度指令 → 車輪回転速度 ---
    def drive(self, v, w):
        v = float(np.clip(v, -V_MAX, V_MAX))
        w = float(np.clip(w, -W_MAX, W_MAX))
        self.data.ctrl[0] = (v - w * WHEEL_L / 2) / WHEEL_R   # 左
        self.data.ctrl[1] = (v + w * WHEEL_L / 2) / WHEEL_R   # 右

    def step(self, n=1, record=True):
        for _ in range(n):
            mujoco.mj_step(self.model, self.data)
            self.t += self.model.opt.timestep
            k = int(round(self.t / self.model.opt.timestep))
            if record and k % self.steps_per_frame == 0:
                self.renderer.update_scene(self.data, camera=self.cam)
                self.frames.append(self.renderer.render().copy())
                self.trace.append((self.t, *self.pose()))

    # --- 中レベル：行動 ---
    def run_for(self, v, w, sec):
        self.drive(v, w)
        self.step(int(sec / self.model.opt.timestep))

    def goto(self, target, margin=0.35, timeout=25.0):
        """目印 target へ向かう。向きを合わせながら進む比例制御。戻り値: 到達したか"""
        tx, ty = LANDMARKS[target]["pos"]
        stop_dist = LANDMARKS[target]["radius"] + margin + 0.2   # 目印の半径 + 余裕 + 車体の長さ
        t0 = self.t
        while self.t - t0 < timeout:
            x, y, yaw = self.pose()
            dx, dy = tx - x, ty - y
            dist = math.hypot(dx, dy)
            if dist < stop_dist:
                self.drive(0, 0); self.step(int(0.3 / self.model.opt.timestep))
                return True
            err = (math.atan2(dy, dx) - yaw + math.pi) % (2 * math.pi) - math.pi   # -pi..pi
            w = 2.0 * err
            v = V_MAX * max(0.0, math.cos(err)) * min(1.0, dist / 0.8)              # 向きがずれていたら減速
            self.drive(v, w)
            self.step(10)
        self.drive(0, 0)
        return False

    def turn(self, deg):
        """その場で deg 度回る（正が左回り）"""
        x, y, yaw0 = self.pose()
        goal = yaw0 + math.radians(deg)
        t0 = self.t
        while self.t - t0 < 10.0:
            _, _, yaw = self.pose()
            err = (goal - yaw + math.pi) % (2 * math.pi) - math.pi
            if abs(err) < math.radians(3):
                break
            self.drive(0, np.clip(2.5 * err, -W_MAX, W_MAX) if abs(err) > 0.2 else np.sign(err) * 0.4)
            self.step(10)
        self.drive(0, 0); self.step(int(0.3 / self.model.opt.timestep))

    def wait(self, sec):
        self.drive(0, 0); self.step(int(sec / self.model.opt.timestep))

    # --- 高レベル：計画の実行 ---
    def execute(self, plan):
        log = []
        for seg in plan:
            a = seg["action"]
            x0, y0, _ = self.pose(); t0 = self.t
            if a == "goto":
                ok = self.goto(seg["target"]); extra = {"reached": ok}
            elif a == "turn":
                self.turn(seg["deg"]); extra = {}
            elif a == "wait":
                self.wait(seg["sec"]); extra = {}
            elif a == "move":   # 前進 sec 秒
                self.run_for(seg.get("v", 0.4), 0.0, seg["sec"]); extra = {}
            else:
                extra = {"error": f"unknown action {a}"}
            x1, y1, yaw1 = self.pose()
            log.append({**seg, **extra, "t_start": round(t0, 2), "t_end": round(self.t, 2),
                        "pos_end": [round(x1, 2), round(y1, 2)], "yaw_end_deg": round(math.degrees(yaw1), 1)})
        return log

    def save_video(self, path):
        imageio.mimwrite(path, self.frames, fps=self.fps, codec="libx264", quality=8, macro_block_size=None)
        return path


if __name__ == "__main__":
    import sys, json, os
    out = sys.argv[1] if len(sys.argv) > 1 else "rover_test.mp4"
    r = Rover()
    plan = [{"action": "goto", "target": "red_pillar"}, {"action": "turn", "deg": 90},
            {"action": "goto", "target": "blue_pillar"}, {"action": "wait", "sec": 1.0},
            {"action": "goto", "target": "green_box"}]
    log = r.execute(plan)
    print(json.dumps(log, ensure_ascii=False, indent=1))
    r.save_video(out)
    print("video:", out, len(r.frames), "frames", os.path.getsize(out) // 1024, "KB")
'''
open('rover_sim.py', 'w').write(rover_src)
print('rover_sim.py を書き出しました')


### 0-4. 動作確認：手書きの計画で走らせて録画（ここまでで合格要件）
赤い柱 → 左に90度 → 青い柱 → 1秒待つ → 緑の箱、という計画を実行します。約20秒の走行で、描画込みで1〜2分かかります。
`reached: True` が並び、動画でロボットが目印を順に回っていれば成功です。


In [ ]:
import importlib, json, rover_sim
importlib.reload(rover_sim)
from rover_sim import Rover, LANDMARKS

def show_video(path, width=640):
    import base64
    from IPython.display import HTML, display
    data = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls loop muted src="data:video/mp4;base64,{data}"></video>'))

test_plan = [{"action": "goto", "target": "red_pillar"}, {"action": "turn", "deg": 90},
             {"action": "goto", "target": "blue_pillar"}, {"action": "wait", "sec": 1.0},
             {"action": "goto", "target": "green_box"}]
r = Rover()
log = r.execute(test_plan)
out = f"{DRIVE}/videos/test_manual.mp4"
r.save_video(out)
for row in log:
    print(row)
print("走行時間 %.1f 秒, フレーム %d" % (r.t, len(r.frames)))
show_video(out)


## 1. ローカルLLM（Qwen3.5）で日本語指示 → 行動計画

### 1-1. モデルの読み込み（初回はダウンロードに数分）
- GPUあり：Qwen3.5-4B を4bit量子化（約3GB）で読み込み。T4はBF16非対応なので計算はfp16
- GPUなし：Qwen3.5-0.8B をCPUで読み込み（1回の応答に10〜40秒）
- Qwen3.5は開発版の `transformers` が必要です。読み込めなかった場合は Qwen3-4B-Instruct-2507（GPU）／Qwen2.5-1.5B-Instruct（CPU）に自動で切り替えます
- Qwen3.5は「思考モード」が既定でオンなので、`enable_thinking=False` で切ります（今回は計画のJSONを出すだけなので不要）


In [ ]:
%pip install -q -U "transformers @ git+https://github.com/huggingface/transformers.git@main" accelerate bitsandbytes

import torch, re, json
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor

USE_GPU = torch.cuda.is_available()
if USE_GPU:
    from transformers import BitsAndBytesConfig
    QUANT = dict(quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                        bnb_4bit_compute_dtype=torch.float16), device_map="auto")
else:
    QUANT = dict(torch_dtype=torch.float32, device_map="cpu")

def load_llm():
    try:
        from transformers import AutoModelForMultimodalLM
        name = "Qwen/Qwen3.5-4B" if USE_GPU else "Qwen/Qwen3.5-0.8B"
        proc = AutoProcessor.from_pretrained(name)
        model = AutoModelForMultimodalLM.from_pretrained(name, **QUANT)
        return name, proc, model, True
    except Exception as e:
        print("Qwen3.5 の読み込みに失敗したため代替モデルに切り替えます:", type(e).__name__, str(e)[:200])
        name = "Qwen/Qwen3-4B-Instruct-2507" if USE_GPU else "Qwen/Qwen2.5-1.5B-Instruct"
        tok = AutoTokenizer.from_pretrained(name)
        model = AutoModelForCausalLM.from_pretrained(name, **QUANT)
        return name, tok, model, False

LLM_NAME, tok_or_proc, llm, IS_VL = load_llm()
print("読み込み完了:", LLM_NAME, "/ GPU" if USE_GPU else "/ CPU")


### 1-2. 計画を作る関数（プロンプト＋検証層）
LLMには「使える行動」と「目印の名前」を教え、JSON配列だけを返させます。返ってきたものは `validate()` で検証します。
- JSONとして読めない → 再生成（最大3回）
- 知らない行動名・知らない目印 → その要素を捨てる
- 角度は±180度、待ち時間は0.5〜5秒に丸める。行動は最大8個
- 検証の結果（再生成回数、捨てた数、丸めた数）は記録し、レポートの考察に使います


In [ ]:
LANDMARK_LIST = "\n".join(f'  "{k}" = {v["ja"]}' for k, v in LANDMARKS.items())
SYSTEM_PROMPT = f"""あなたは移動ロボットの行動計画係です。ユーザーの日本語の指示を、ロボットが実行できる行動の列に変換します。
出力はJSON配列のみ。説明文は書かない。使える行動は次の4種類だけ:
  {{"action": "goto", "target": "<目印の名前>"}}   目印まで移動して手前で止まる
  {{"action": "turn", "deg": <角度>}}              その場で回る。正が左回り、負が右回り（例: 右に90度 → -90）
  {{"action": "wait", "sec": <秒>}}                その場で待つ
  {{"action": "move", "sec": <秒>}}                前に進む
目印の名前（英語の名前をそのまま使う）:
{LANDMARK_LIST}
例: 指示「赤い柱に行って、右を向いて、青い柱へ行って」
[{{"action": "goto", "target": "red_pillar"}}, {{"action": "turn", "deg": -90}}, {{"action": "goto", "target": "blue_pillar"}}]"""

def llm_generate(user_text, max_new_tokens=200):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_text}]
    if IS_VL:
        inputs = tok_or_proc.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True,
                                                 return_tensors="pt", enable_thinking=False).to(llm.device)
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        text = tok_or_proc.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    else:
        prompt = tok_or_proc.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, enable_thinking=False)
        inputs = tok_or_proc(prompt, return_tensors="pt").to(llm.device)
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        text = tok_or_proc.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()

def validate(raw, max_actions=8):
    m = re.search(r"\[.*\]", raw, flags=re.S)
    if not m:
        raise ValueError("JSON配列が見つからない")
    plan = json.loads(m.group(0))
    if not isinstance(plan, list) or not plan:
        raise ValueError("空、または配列でない")
    clean, dropped, clamped = [], 0, 0
    for seg in plan[:max_actions]:
        a = seg.get("action")
        if a == "goto" and seg.get("target") in LANDMARKS:
            clean.append({"action": "goto", "target": seg["target"]})
        elif a == "turn":
            d = float(seg.get("deg", 90)); c = max(-180.0, min(180.0, d)); clamped += (c != d)
            clean.append({"action": "turn", "deg": c})
        elif a in ("wait", "move"):
            s = float(seg.get("sec", 1.0)); c = max(0.5, min(5.0, s)); clamped += (c != s)
            clean.append({"action": a, "sec": c})
        else:
            dropped += 1
    if not clean:
        raise ValueError("実行できる行動がない")
    return clean, dropped, clamped

def plan_from_instruction(text, retries=3):
    raw, last = "", None
    for i in range(retries):
        raw = llm_generate(text)
        try:
            plan, dropped, clamped = validate(raw)
            return {"instruction": text, "plan": plan, "raw": raw, "retries": i, "dropped": dropped, "clamped": clamped}
        except (ValueError, json.JSONDecodeError) as e:
            last = f"{type(e).__name__}: {e}"
    return {"instruction": text, "plan": None, "raw": raw, "retries": retries, "error": last}

r = plan_from_instruction("緑の箱まで行って、そこで右に90度向いて、2秒待ってから黄色い箱へ")
print(json.dumps(r, ensure_ascii=False, indent=1))


### 1-3. 日本語指示 → 計画 → 走行 → 録画（本番）
`INSTRUCTIONS` を書き換えれば好きな指示で試せます。言い回しを変えた指示や、わざと存在しない目印（「紫の柱」）を混ぜると、検証層が働く様子を記録できます。
各指示について、動画を `videos/`、計画と走行記録を `plans/` に保存します。最後の表はレポートの「結果」に使えます。


In [ ]:
INSTRUCTIONS = [
    "赤い柱まで行って止まって",
    "青い柱に行ってから、左を向いて、緑の箱まで行って",
    "黄色い箱の前で3秒待ってから、赤い柱へ向かって",
    "紫の柱に行ってから青い柱へ行って",          # 存在しない目印 → 検証層が捨てる
    "まず緑の箱、次に黄色い箱、最後に赤い柱を順番に回って",
]

results = []
for i, text in enumerate(INSTRUCTIONS):
    res = plan_from_instruction(text)
    if res["plan"] is None:
        print(f"[{i}] 計画生成に失敗:", res["error"]); results.append(res); continue
    rv = Rover()
    res["log"] = rv.execute(res["plan"])
    res["video"] = f"{DRIVE}/videos/inst{i:02d}.mp4"
    rv.save_video(res["video"])
    res["duration_sec"] = round(rv.t, 1)
    json.dump(res, open(f"{DRIVE}/plans/inst{i:02d}.json", "w"), ensure_ascii=False, indent=1)
    results.append(res)
    print(f"[{i}] {text}\n     計画: {res['plan']}\n     再生成 {res['retries']} / 捨てた {res['dropped']} / 丸め {res['clamped']} / 走行 {res['duration_sec']} 秒")

import pandas as pd
rows = [{"指示": x["instruction"], "行動数": len(x["plan"]) if x["plan"] else 0, "再生成": x["retries"],
         "捨てた": x.get("dropped"), "丸め": x.get("clamped"),
         "到達": sum(1 for s in x.get("log", []) if s.get("reached")), "走行秒": x.get("duration_sec")} for x in results]
display(pd.DataFrame(rows))
show_video(results[1]["video"]) if results[1].get("video") else None


## 2. 提出用の動画に仕上げる
各クリップの前に「指示文」と「LLMが出した計画」のカードを挟み、1本の mp4 につなぎます。
合計が3分を超えるときは `SPEED = 2.0` にして2倍速にすると、冒頭カードに「2倍速」と入ります（課題の要件）。
**氏名・所属・アカウント名は入れない**でください。


In [ ]:
from PIL import Image, ImageDraw, ImageFont
import subprocess, textwrap

SPEED = 1.0          # 3分を超えるなら 2.0
TITLE = "日本語指示で目印を巡回する移動ロボット\nMuJoCo × ローカルLLM（" + LLM_NAME.split("/")[-1] + "）"
FONT = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
W, H, FPS = 640, 480, 25
work = "/content/final_video"; os.makedirs(work, exist_ok=True)

def card(text, path, sec=3.0, size=26):
    img = Image.new("RGB", (W, H), (24, 28, 32)); d = ImageDraw.Draw(img)
    font = ImageFont.truetype(FONT, size)
    lines = [l for t in text.split("\n") for l in (textwrap.wrap(t, 22) or [""])]
    y = H // 2 - len(lines) * (size + 10) // 2
    for l in lines:
        w = d.textlength(l, font=font); d.text(((W - w) / 2, y), l, font=font, fill=(240, 240, 240)); y += size + 10
    png = path.replace(".mp4", ".png"); img.save(png)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-loop", "1", "-i", png, "-t", str(sec), "-r", str(FPS),
                    "-vf", f"scale={W}:{H}", "-pix_fmt", "yuv420p", path], check=True)
    return path

def plan_text(plan):
    ja = {"goto": lambda s: LANDMARKS[s["target"]]["ja"] + "へ", "turn": lambda s: f"{s['deg']:+.0f}度回る",
          "wait": lambda s: f"{s['sec']:.0f}秒待つ", "move": lambda s: f"{s['sec']:.0f}秒前進"}
    return " → ".join(ja[s["action"]](s) for s in plan)

parts = [card(TITLE + ("\n（%g倍速）" % SPEED if SPEED != 1 else ""), f"{work}/00_title.mp4", 4.0)]
for i, x in enumerate(results):
    if not x.get("video"): continue
    parts.append(card("指示：" + x["instruction"] + "\n\nLLMの計画：" + plan_text(x["plan"]), f"{work}/{i:02d}_card.mp4", 3.5, 22))
    norm = f"{work}/{i:02d}_clip.mp4"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", x["video"], "-vf", f"setpts={1/SPEED}*PTS,scale={W}:{H}",
                    "-r", str(FPS), "-pix_fmt", "yuv420p", "-an", norm], check=True)
    parts.append(norm)
open(f"{work}/list.txt", "w").write("".join(f"file '{p}'\n" for p in parts))
FINAL = f"{DRIVE}/videos/PAI_final_submission.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", f"{work}/list.txt",
                "-c:v", "libx264", "-pix_fmt", "yuv420p", "-r", str(FPS), FINAL], check=True)
dur = float(subprocess.check_output(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", FINAL]))
print(f"完成: {FINAL}  長さ {dur:.1f} 秒（要件: 60〜180秒。超えるなら SPEED=2.0）")
show_video(FINAL)


## 3. 提出前チェック
- 動画：60〜180秒、倍速なら明記、個人情報なし → YouTube（限定公開以上、タイトル `PAI最終課題_アカウントID`）
- コード：このノート（メニュー「編集」→「すべての出力を消去」してから保存）、`rover_sim.py`、`plans/*.json` を GitHub の Public リポジトリへ
- レポートPDF：`説明資料_PAI最終課題_アカウント名.pdf`。目的／構成図／工夫点（責任分離・検証層・ローカルLLM）／結果の表／反省点（Genesis強化学習の試みとクラッシュ、切り替えの判断）
- Driveのパスやアカウント名がノートの出力に残っていないか確認
